In [7]:
import pandas as pd
import json
import requests
import time

In [3]:
data = pd.read_excel('batch_v1_size_1000.xlsx')
data.head(3)

,Unnamed: 0,id,hh_vac_id,hh_vac_link,title,experience,employment,schedule,salary_raw,salary_from,salary_to,currency,employer,address_raw,area,skills,published_at,raw_page
0,0,1,131963477,https://hh.ru/vacancy/131963477,Senior AI Multiagent Systems Engineer,Опыт более 6 лет,NaN,NaN,"400 000 – 700 000₽за месяц,на руки",NaN,NaN,NaN,Simplenight,Москва,NaN,NaN,NaN,"<!DOCTYPE html>\n<html class=""desktop"" lang=""r..."
1,1,2,132530194,https://hh.ru/vacancy/132530194,Middle+ AI-инженер,Опыт 1-3 года,NaN,NaN,"до320 000₽за месяц,до вычета налогов",NaN,NaN,NaN,Точка Банк,Москва,NaN,[],2026-04-27T18:53:55.340855,NaN
2,2,3,131744414,https://hh.ru/vacancy/131744414,Prompt Engineer / AI‑редактор (нейросети и авт...,Опыт 1-3 года,полная,полный день,"80 000 – 100 000₽за месяц,на руки",80000.0,100000.0,RUR,ДЖЕЙКЕТ РАБОТА,Москва,Москва,[],2026-04-27T18:54:50.686408,"<!DOCTYPE html>\n<html class=""desktop"" lang=""r..."


In [14]:
PROMPT = """
Ты — промышленная система извлечения структурированных данных из вакансий.
Твоя задача — максимально точно преобразовать неструктурированный текст вакансии в строгий JSON.

==================================================
ОСНОВНАЯ ЦЕЛЬ
==================================================

На вход подаются поля вакансии:

- title
- experience
- employment
- schedule
- salary_raw
- employer
- area
- text / description / raw_page_text

Нужно вернуть JSON:

{
  "skills": [],
  "level": "unknown",
  "category": "Other",
  "salary_from": null,
  "salary_to": null,
  "currency": "unknown"
}

==================================================
ОБЩИЕ ПРАВИЛА
==================================================

1. Используй ТОЛЬКО информацию из входных данных.
2. Не выдумывай требования.
3. Если данных нет — ставь null / unknown / [].
4. Ответ только JSON.
5. Никакого markdown.
6. Никаких комментариев.
7. Никакого текста до JSON и после JSON.
8. Все ключи обязательны.
9. JSON должен быть валиден для json.loads().
10. Если сомневаешься — выбирай консервативный вариант.

==================================================
ПОЛЕ 1: skills
==================================================

Верни массив профессиональных навыков кандидата.

Допускаются:

A. IT:
Python
SQL
Pandas
NumPy
Docker
Kubernetes
Linux
Git
FastAPI
Django
Flask
JavaScript
React
Node.js
AWS
GCP
Azure
TensorFlow
PyTorch
LLM
RAG
LangChain
LangGraph
OpenAI API
Prompt Engineering

B. Аналитика:
Excel
Power BI
Tableau
1C
Google Sheets
A/B Testing
SQL

C. Бухгалтерия / финансы:
Бухгалтерский учет
Налоговый учет
МСФО
1С Бухгалтерия
Финансовый анализ
Бюджетирование

D. Продажи:
CRM
B2B продажи
B2C продажи
Холодные продажи
Активные продажи
Переговоры
Работа с возражениями

E. Маркетинг:
SEO
SMM
Контекстная реклама
Яндекс Директ
Google Ads
Email-маркетинг
Контент-маркетинг

F. HR:
Подбор персонала
Boolean Search
Interviewing
Кадровое делопроизводство
Онбординг

G. Производство / логистика:
SAP
Закупки
Логистика
Складской учет
Контроль качества

H. Юриспруденция:
Договорная работа
Корпоративное право
Комплаенс
Судебная практика

I. Медицина:
Сестринское дело
Фармакология
УЗИ
Медицинская документация

-----------------------------------
НЕЛЬЗЯ добавлять:
-----------------------------------

ответственность
коммуникабельность
стрессоустойчивость
пунктуальность
инициативность
обучаемость
работа в команде
многозадачность

Это soft skills.

-----------------------------------
Правила:
-----------------------------------

1. Только подтвержденные вакансией навыки.
2. Если навык явно в title — можно использовать.
3. Нормализуй названия.
4. Убирай дубли.
5. Максимум 15 навыков.
6. Если навыки неясны — [].

==================================================
ПОЛЕ 2: level
==================================================

Одно значение:

junior
middle
senior
lead
head
unknown

Правила:

стажер / intern / trainee / без опыта -> junior
1-3 года -> junior или middle
3-6 лет -> middle или senior
6+ лет -> senior или lead
Lead -> lead
Head / Руководитель / Директор направления -> head

Если роль senior в title => senior
Если middle в title => middle
Если junior в title => junior

Если нет данных => unknown

==================================================
ПОЛЕ 3: category
==================================================

Одно значение:

AI Engineer
Software Developer
Backend Developer
Frontend Developer
Fullstack Developer
Data Scientist
ML Engineer
Data Engineer
DevOps Engineer
QA Engineer
Mobile Developer
Accountant
Financial Analyst
Sales Manager
Marketing Manager
HR Specialist
Recruiter
Lawyer
Doctor
Nurse
Designer
Product Manager
Project Manager
Business Analyst
Operations Manager
Logistics Specialist
Teacher
Customer Support
Administrator
Engineer
Other

Правила:

LLM / AI agents / Prompt / RAG -> AI Engineer
Python backend / Java / Go / C# -> Backend Developer
React / Vue -> Frontend Developer
Fullstack -> Fullstack Developer
ML / model training -> ML Engineer
Analytics + ML -> Data Scientist
Pipelines / ETL / warehouse -> Data Engineer
CI/CD / infra / k8s -> DevOps Engineer

Если неясно -> Other

==================================================
ПОЛЯ 4-6: salary_from salary_to currency
==================================================

Используй salary_raw.

Примеры:

400 000 – 700 000 ₽
до 320 000 ₽
от 250 000 руб
3000 USD
1500-2500 EUR
100000
зп не указана

Правила:

1. currency:

₽ руб руб. RUR RUB -> RUR
$ USD USDT? только если явно зарплата -> USD
€ EUR -> EUR
₸ -> KZT
BYN -> BYN
иначе unknown

2. Диапазон:

400000-700000
400000 – 700000
от 400000 до 700000

=> salary_from=400000
salary_to=700000

3. Только верхняя граница:

до 320000

=> salary_from=null
salary_to=320000

4. Только нижняя:

от 250000

=> salary_from=250000
salary_to=null

5. Одно число:

100000

=> salary_from=100000
salary_to=100000

6. Если не указано:

salary_from=null
salary_to=null
currency="unknown"

7. Удаляй пробелы, неразрывные пробелы, символы валют.

8. Только числа, без строк.

==================================================
ПРОВЕРКА ПЕРЕД ОТВЕТОМ
==================================================

Убедись что:

- JSON валиден
- Все ключи присутствуют
- skills это массив
- salary_from число или null
- salary_to число или null
- currency строка
- level из допустимого списка
- category из допустимого списка

==================================================
ФОРМАТ ОТВЕТА
==================================================

{
  "skills": ["Python", "SQL"],
  "level": "middle",
  "category": "AI Engineer",
  "salary_from": 250000,
  "salary_to": 400000,
  "currency": "RUR"
}
"""

In [19]:
API_KEY = "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"
# MODEL = "openai/gpt-4o-mini"
MODEL = "deepseek/deepseek-v3.2"

def enrich_vacancy(row):

    prompt = f"""
    Вакансия:
    title: {row['title']}
    experience: {row['experience']}
    salary: {row['salary_raw']}
    employer: {row['employer']}
    area: {row['area']}
    text: {str(row['raw_page'])[:4000]}
    """

    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }

    data = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": PROMPT},
            {"role": "user", "content": prompt}
        ],
        "response_format": {"type": "json_object"}
    }

    r = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers=headers,
        json=data
    )

    txt = r.json()["choices"][0]["message"]["content"]

    try:
        print(txt)
        return json.loads(txt)
    except:
        return {"skills": [], "level": None, "category": None}

In [20]:
results = []

for _, row in data[554:].iterrows():
    print(f'Начало : {_}')

    res = enrich_vacancy(row)
    results.append(res)
    time.sleep(0.05)

    print(f'Конец : {_}')

    if _ == 5000:
        break


extra = pd.DataFrame(results)
extra.to_excel('extra.xlsx')

df = pd.concat([data, extra], axis=1)
df.to_excel('df.xlsx')

Начало : 554


KeyError: 'choices'

In [16]:
extra = pd.DataFrame(results)
extra.to_excel('extra.xlsx')

df = pd.concat([data, extra], axis=1)
df.to_excel('df.xlsx')